# RoleScout — Agentic Job Hunt Intelligence System

**What it does:** Takes a raw job description and a candidate profile, runs three specialized AI agents in parallel to research the company, the role, and candidate fit, then synthesizes everything into a structured interview prep report.

---

## Architecture

```mermaid
flowchart TD
  START((START)) --> ParseJD[Parse Job Description]
  ParseJD --> HumanFeedback[Human Feedback]
  HumanFeedback -- retry --> ParseJD
  HumanFeedback -- continue --> Spawn[spawn]
  Spawn --> Company[Company Agent]
  Spawn --> Role[Role Agent]
  Spawn --> Fit[Fit Analyzer]
  Company --> Synthesize[Synthesize]
  Role --> Synthesize
  Fit --> Synthesize
  Synthesize --> END((END))
```

**Key patterns used:**
- Structured output (Pydantic) for reliable LLM extraction
- Human-in-the-loop (HITL) with `interrupt_before` for JD correction
- MapReduce parallelisation via the `Send` API
- ReAct loops inside each research agent (tool call → observe → repeat)
- MemorySaver checkpointing for cross-cell state persistence

---

## Sections
0. Environment Setup
1. Core Configuration
2. State Schema
3. JD Parsing Agent
4. Research Agents
5. Synthesis Agent
6. Graph Orchestration
7. Run

## 0. Environment Setup

Loads API keys from `.env` and enables LangSmith tracing for observability.

In [ ]:
import os
import operator
import json
from enum import Enum
from typing import List, Literal, Annotated
from datetime import datetime

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from IPython.display import Image, display

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Send

load_dotenv()

# LangSmith tracing — set to 'false' to disable
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT", "rolescout")

## 1. Core Configuration

Initialise the LLM. Currently using OpenRouter with `nvidia/nemotron` (free tier). 
Swap the model string to use any OpenRouter-supported model.

In [ ]:
from langchain_openrouter import ChatOpenRouter

MODEL = os.getenv("MODEL", "nvidia/nemotron-3-super-120b-a12b:free")

llm = ChatOpenRouter(model=MODEL, temperature=0.0)

## 2. State Schema

### 2a. JD Parsing Schemas
Structured output targets for the JD parser.

In [ ]:
class Skills(BaseModel):
    """A single skill extracted from a job description."""
    name: str
    level: str | None = None
    mandatory: bool = True


class RequiredSkills(BaseModel):
    """Classified skill set from the JD."""
    hard_skills: List[Skills]
    soft_skills: List[Skills]


class Flag(BaseModel):
    """A hiring signal extracted from the JD."""
    flag_type: Literal["Red", "Green"]
    desc: str


class JDParserOutput(BaseModel):
    """Structured output of the JD parsing agent."""
    company_name: str
    role_title: str
    required_skills: RequiredSkills
    seniority_level: Literal[
        "Intern", "Junior", "Mid-Level",
        "Senior", "Staff", "Principal", "Leadership"
    ]
    flags: List[Flag]

### 2b. Candidate Profile Schemas

Used by the Fit Analyzer. Fill in your profile in Section 7 (Run).

In [ ]:
class Skill(BaseModel):
    """A candidate's skill with self-assessed proficiency (1–10)."""
    name: str
    proficiency: int = Field(ge=1, le=10)


class MySkill(BaseModel):
    hard_skills: List[Skill]
    soft_skills: List[Skill]


class DeploymentStatus(str, Enum):
    """Deployment tier of a project. Higher = more weight in fit scoring."""
    NOT_DEPLOYED = "not_deployed"
    INTERNAL = "internal"
    PUBLIC = "public"
    PRODUCTION = "production"


class Project(BaseModel):
    name: str
    tech_stack: List[str]
    deployment_status: DeploymentStatus


class Profile(BaseModel):
    """Candidate profile passed into the graph at invocation time."""
    name: str
    skills: MySkill
    experience_years: int
    projects: List[Project]
    preferred_roles: List[str]
    location: str

### 2c. Research Agent Output Schemas

Each parallel agent returns one of these. All three get appended to `ScoutState.results`.

In [ ]:
class CompanyResearcher(BaseModel):
    """Output of the Company Research agent."""
    company_name: str
    what_they_do: str
    recent_news: List[str]
    tech_stack: List[str]
    culture_signals: List[str]
    red_flags: List[str]
    funding_projects: List[str]
    product_launches: List[str]
    glassdoor_culture_review: List[str]
    leadership_changes: List[str]
    layoffs: List[str]


class RoleResearcher(BaseModel):
    """Output of the Role Research agent."""
    actual_responsibilities: List[str]
    must_have_skills: List[str]
    nice_to_have_skills: List[str]
    typical_interview_format: List[str]
    salary_range: str


class FitAnalyzer(BaseModel):
    """Output of the Fit Analyzer agent."""
    strong_matches: List[str]
    gaps: List[str]
    talking_points: List[str]
    fit_score: int = Field(ge=1, le=10)

### 2d. Memory Schemas

Used for long-term persistence across sessions (v2 feature).

- `ApplicationHistory` — persisted after every research session
- `WeakArea` — populated after interview practice sessions (v2)

In [ ]:
class ApplicationHistory(BaseModel):
    """Persisted record of a researched role. Stored in long-term memory."""
    company_name: str
    role_title: str
    seniority_level: str
    overall_fit_score: int = Field(ge=1, le=10)
    technical_fit_score: int = Field(ge=1, le=10)
    experience_fit_score: int = Field(ge=1, le=10)
    missing_skills: List[str] = []
    strong_matches: List[str] = []
    key_gaps: List[str] = []
    aligned_projects: List[str] = []
    company_tier: str | None = None       # startup / mid-size / big-tech / research
    compensation_signal: str | None = None  # low / medium / high / unknown
    notes: str | None = None
    researched_at: datetime


class WeakArea(BaseModel):
    """Tracked weakness from interview practice sessions (v2)."""
    # TODO: v2 - add fields: question_type, frequency, pattern, last_seen
    pass

### 2e. Graph State

`ScoutState` is the single shared object passed through the entire graph.

- `results` uses `operator.add` as a reducer so parallel agents can safely append simultaneously without overwriting each other.

In [ ]:
class ScoutState(BaseModel):
    """Shared state passed through the entire RoleScout graph."""
    job_desc: str
    profile: Profile
    parsed_jd: JDParserOutput | None = None
    human_feedback: str | None = None
    # operator.add reducer: parallel agents append without overwriting
    results: Annotated[list[dict], operator.add] = Field(default_factory=list)
    final_report: str | None = None

## 3. JD Parsing Agent

Node 1: Extracts structured information from a raw job description.
Node 2: No-op HITL node — graph pauses here for human correction.

If `human_feedback` is present → re-parses the JD with the correction injected.
If `human_feedback` is None → continues to parallel research.

In [ ]:
PARSER_INSTRUCTION = """
You are an expert hiring intelligence and job description parsing system.

Your task is to accurately extract structured hiring information from the provided job description.

JOB DESCRIPTION:
{job_desc}

OPTIONAL HUMAN FEEDBACK:
{human_feedback}

Instructions:

1. Extract the company name exactly as mentioned.
2. Extract the role title exactly as mentioned.

3. Determine the seniority level based on:
   - years of experience required
   - ownership expectations
   - technical depth
   - leadership expectations
   - scope of responsibilities

   Allowed values: Intern | Junior | Mid-Level | Senior | Staff | Principal | Leadership

4. Extract required skills, classified as hard_skills or soft_skills.
   - Normalise skill names (e.g. 'LangChain / LangGraph' → separate entries)
   - Avoid duplicates
   - Set mandatory=True only if explicitly required or strongly implied

5. Generate hiring flags.
   Green: clear technical expectations, realistic scope, focused tech stack, well-defined responsibilities
   Red: unrealistic expectations, contradictory requirements, excessive tech breadth, vague responsibilities

6. Do not hallucinate. Prefer precision over completeness.
   If uncertain, omit rather than guess.
"""


def parse_jd(state: ScoutState) -> dict:
    """Parses a raw job description into a structured JDParserOutput."""
    structured_llm = llm.with_structured_output(JDParserOutput)
    system_message = PARSER_INSTRUCTION.format(
        job_desc=state.job_desc,
        human_feedback=state.human_feedback or "None provided."
    ) + "\nReturn ONLY valid JSON matching the schema."

    jd_parsed = structured_llm.invoke([
        SystemMessage(content=system_message),
        HumanMessage(content="Parse the job description and return JSON only.")
    ])

    return {"parsed_jd": jd_parsed.model_dump()}


def human_feedback(state: ScoutState) -> dict:
    """No-op node. Graph is interrupted here for human correction via graph.invoke(None)."""
    return {}


def should_continue(state: ScoutState) -> str:
    """Routes back to parsing if human feedback was provided, otherwise continues."""
    if state.human_feedback:
        return "retry"
    return "continue"

## 4. Research Agents

### 4a. Search Tools

Two tools available to all research agents:
- `web_search` — general background, tech stack, leadership
- `search_news` — recent news, funding, layoffs (falls back to web search if news unavailable)

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException


@tool
def web_search(query: str) -> list[dict]:
    """
    Search the web for general information.

    Use for: company background, tech stack, role expectations,
    interview patterns, salary benchmarks, leadership info.

    Args:
        query: Specific search query. Example: 'Databricks AI engineer tech stack 2025'

    Returns:
        List of results with title, url, and body text.
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
            return results if results else []
    except DDGSException:
        return []
    except Exception as e:
        return [{"error": str(e), "source": "web_search"}]


@tool
def search_news(query: str) -> list[dict]:
    """
    Search for recent news articles on any topic.

    Use for: company updates, funding rounds, layoffs, hiring trends,
    market movement, leadership changes, salary shifts.

    Args:
        query: Specific news query. Example: 'Databricks funding 2025'

    Returns:
        List of news results with title, url, date, and body text.
        Falls back to web search if news results are unavailable.
    """
    try:
        with DDGS() as ddgs:
            news = list(ddgs.news(query, max_results=5))
            if news:
                return news
    except (DDGSException, Exception):
        pass

    # Fallback to web search if news API returns nothing
    try:
        with DDGS() as ddgs:
            fallback = list(ddgs.text(query, max_results=5))
            return fallback if fallback else []
    except Exception as e:
        return [{"error": str(e), "source": "search_news"}]


tools = [web_search, search_news]

### 4b. Company Researcher

ReAct loop: searches for company background, news, culture, funding, and tech stack.
Capped at 5 tool call rounds to control cost. Structures output into `CompanyResearcher`.

In [ ]:
COMPANY_RESEARCHER_INSTRUCTION = """
You are a Company Research Agent. Your job is to thoroughly research a company for a job applicant.

Tools available:
- web_search: general info, tech stack, products, leadership, about pages
- search_news: recent news, funding rounds, layoffs, leadership changes (last 6 months)

Research targets:
- What the company does (core product/service, 2-3 sentences)
- Recent news relevant to engineers (last 6 months)
- Tech stack they use publicly
- Culture signals (work environment, values, employee sentiment)
- Red flags (layoffs, controversies, bad reviews)
- Funding history and active projects
- Recent product launches
- Glassdoor culture reviews summary
- Leadership changes
- Any layoffs or hiring freezes

Run multiple searches with different queries. Do not stop after one search.
Aim for at least 3 searches: one general, one news, one culture/glassdoor.

Company to research: {company_name}
"""


def company_researcher(state: ScoutState) -> dict:
    """Runs a ReAct research loop on the company, returns structured CompanyResearcher output."""
    llm_with_tools = llm.bind_tools(tools)
    messages = [
        SystemMessage(content=COMPANY_RESEARCHER_INSTRUCTION.format(
            company_name=state.parsed_jd.company_name
        )),
        HumanMessage(content=f"Research this company: {state.parsed_jd.company_name}")
    ]

    # ReAct loop: tool call → observe → repeat until done or cap reached
    for _ in range(5):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            break

        for tool_call in response.tool_calls:
            selected_tool = web_search if tool_call["name"] == "web_search" else search_news
            result = selected_tool.invoke(tool_call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

    # Structure the gathered information
    structured_result = llm.with_structured_output(CompanyResearcher).invoke(
        messages + [HumanMessage(content="Structure findings. Return JSON only.")]
    )
    return {"results": [structured_result.model_dump()]}

### 4c. Role Researcher

ReAct loop: researches what this role actually involves beyond the JD, including
real responsibilities, interview format, salary benchmarks, and in-demand status.

In [ ]:
ROLE_RESEARCHER_INSTRUCTION = """
You are a Role Research Agent. Your job is to research what a specific role actually involves
at a company, beyond what the job description says.

Tools available:
- web_search: role responsibilities, required skills, interview formats
- search_news: role demand trends, salary changes, industry shifts

Research targets:
- Day-to-day responsibilities at this seniority level
- Skills that truly matter vs. JD filler
- Typical interview format (technical rounds, system design, behavioral)
- Salary benchmarks for this role and seniority level
- How in-demand this role is right now
- How this role differs at this company vs. industry standard

Run multiple searches with different queries. Do not stop after one search.

Role: {role_title}
Company: {company_name}
Seniority: {seniority}
"""


def role_researcher(state: ScoutState) -> dict:
    """Runs a ReAct research loop on the role, returns structured RoleResearcher output."""
    llm_with_tools = llm.bind_tools(tools)
    messages = [
        SystemMessage(content=ROLE_RESEARCHER_INSTRUCTION.format(
            role_title=state.parsed_jd.role_title,
            company_name=state.parsed_jd.company_name,
            seniority=state.parsed_jd.seniority_level
        )),
        HumanMessage(content=f"Research this role: {state.parsed_jd.role_title}")
    ]

    for _ in range(5):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            break

        for tool_call in response.tool_calls:
            selected_tool = web_search if tool_call["name"] == "web_search" else search_news
            result = selected_tool.invoke(tool_call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

    structured_result = llm.with_structured_output(RoleResearcher).invoke(
        messages + [HumanMessage(content="Structure findings. Return valid JSON only.")]
    )
    return {"results": [structured_result.model_dump()]}

### 4d. Fit Analyzer

No search tools. Reads the parsed JD and candidate profile directly from state.
Compares skills, experience, projects, and seniority to produce a fit score and gap analysis.

Scoring rules:
- Production deployments weighted higher than internal
- Missing mandatory skills are penalised
- Conservative scoring — prefers underscoring to overscoring

In [ ]:
FIT_ANALYZER_INSTRUCTION = """
You are an expert hiring intelligence and candidate fit evaluation system.

Your task is to evaluate how well the candidate profile matches the target role.

--- JOB INFORMATION ---
Company: {company_name}
Role: {role_title}
Target Seniority: {seniority}
Required Skills: {required_skills}

--- CANDIDATE PROFILE ---
Name: {candidate_name}
Experience: {experience_years} years
Preferred Roles: {preferred_roles}
Location: {location}
Hard Skills: {hard_skills}
Soft Skills: {soft_skills}
Projects: {projects}

--- EVALUATION RULES ---
1. Compare skills against requirements (technical overlap, experience, projects, seniority, role fit).
2. Project deployment weight: production > public > internal > not_deployed.
3. Penalise: missing mandatory skills, seniority mismatch, weak project evidence, low experience.
4. Do not hallucinate experience or skills not in the profile.
5. Prefer conservative scoring. A 4/10 with accurate gaps beats an inflated 7/10.
6. Output strictly following the provided schema.
"""


def fit_analyzer(state: ScoutState) -> dict:
    """Compares candidate profile against JD requirements. No search tools — reads from state only."""
    parsed_jd = state.parsed_jd
    profile = state.profile

    instruction = FIT_ANALYZER_INSTRUCTION.format(
        company_name=parsed_jd.company_name,
        role_title=parsed_jd.role_title,
        seniority=parsed_jd.seniority_level,
        required_skills=parsed_jd.required_skills,
        candidate_name=profile.name,
        experience_years=profile.experience_years,
        preferred_roles=profile.preferred_roles,
        location=profile.location,
        hard_skills=profile.skills.hard_skills,
        soft_skills=profile.skills.soft_skills,
        projects=profile.projects
    )

    result = llm.with_structured_output(FitAnalyzer).invoke([
        SystemMessage(content=instruction),
        HumanMessage(content="Evaluate candidate fit. Return JSON only.")
    ])

    return {"results": [result.model_dump()]}

## 5. Synthesis Agent

Receives outputs from all three parallel agents via `state.results`.
Identifies each agent's output by its unique keys, then synthesizes everything
into a structured interview prep report.

In [ ]:
SYNTHESIZER_INSTRUCTION = """
You are a Job Hunt Intelligence Agent.
You have received research from three specialized agents. Synthesize everything into one structured report.

--- PARSED JOB DESCRIPTION ---
Company: {company_name}
Role: {role_title}
Seniority: {seniority}

--- COMPANY RESEARCH ---
{company_brief}

--- ROLE RESEARCH ---
{role_brief}

--- FIT ANALYSIS ---
{fit_analysis}

Generate a structured report with exactly these sections:

1. COMPANY SNAPSHOT
   - What they do (2-3 sentences)
   - Culture signals
   - Red flags (if any)
   - Recent news that matters for engineers

2. ROLE REALITY CHECK
   - What this role actually involves vs what the JD says
   - Skills that truly matter
   - Typical interview format

3. YOUR FIT
   - Fit score: {fit_score}/10
   - Strong matches
   - Gaps to address
   - Talking points for your experience

4. INTERVIEW PREP
   - 5 technical questions likely to be asked
   - 3 behavioral questions based on their culture
   - 2 questions YOU should ask them

5. BOTTOM LINE
   - Should you apply? Why?
   - Top 3 things to prepare before the interview
"""


def synthesizer(state: ScoutState) -> dict:
    """Synthesizes outputs from all three research agents into a final report."""
    # Identify each agent's output by its unique keys
    company_brief = next((r for r in state.results if "company_name" in r), {})
    role_brief = next((r for r in state.results if "actual_responsibilities" in r), {})
    fit_analysis = next((r for r in state.results if "fit_score" in r), {})

    messages = [
        SystemMessage(content=SYNTHESIZER_INSTRUCTION.format(
            company_name=state.parsed_jd.company_name,
            role_title=state.parsed_jd.role_title,
            seniority=state.parsed_jd.seniority_level,
            company_brief=company_brief,
            role_brief=role_brief,
            fit_analysis=fit_analysis,
            fit_score=fit_analysis.get("fit_score", "N/A")
        )),
        HumanMessage(content="Generate the full synthesis report now.")
    ]

    response = llm.invoke(messages)
    return {"final_report": response.content}

## 6. Graph Orchestration

Wires all nodes together.

Key decisions:
- `interrupt_before=['Human Feedback']` — pauses for user correction after parsing
- `spawn` is a passthrough node; `spawn_researchers` is the routing function that fires `Send` calls
- All three research agents fan back into `synthesize` — LangGraph waits for all three before proceeding

In [ ]:
def spawn_node(state: ScoutState) -> ScoutState:
    """Passthrough node. Routing happens in spawn_researchers."""
    return state


def spawn_researchers(state: ScoutState) -> list:
    """Fires three parallel Send calls — one per research agent."""
    return [
        Send("company_researcher", state),
        Send("role_researcher", state),
        Send("fit_analyzer", state)
    ]


# Build graph
builder = StateGraph(ScoutState)

builder.add_node("Parse Job Description", parse_jd)
builder.add_node("Human Feedback", human_feedback)
builder.add_node("spawn", spawn_node)
builder.add_node("company_researcher", company_researcher)
builder.add_node("role_researcher", role_researcher)
builder.add_node("fit_analyzer", fit_analyzer)
builder.add_node("synthesize", synthesizer)

builder.add_edge(START, "Parse Job Description")
builder.add_edge("Parse Job Description", "Human Feedback")
builder.add_conditional_edges(
    "Human Feedback",
    should_continue,
    {"retry": "Parse Job Description", "continue": "spawn"}
)
builder.add_conditional_edges(
    "spawn",
    spawn_researchers,
    ["company_researcher", "role_researcher", "fit_analyzer"]
)
builder.add_edge("company_researcher", "synthesize")
builder.add_edge("role_researcher", "synthesize")
builder.add_edge("fit_analyzer", "synthesize")
builder.add_edge("synthesize", END)

memory = MemorySaver()
graph = builder.compile(interrupt_before=["Human Feedback"], checkpointer=memory)

display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

## 7. Run

### Step 1: Initial invoke
Paste any job description and fill in your profile. Graph runs until the HITL interrupt.

In [ ]:
thread = {"configurable": {"thread_id": "rolescout-session-1"}}

JOB_DESCRIPTION = """
Company: Databricks

Role: Senior AI Engineer

Location: Bengaluru, India

Experience: 4-7 Years

Responsibilities:
- Build production LLM applications
- Develop RAG pipelines
- Design LangGraph workflows
- Work with vector databases
- Build APIs using FastAPI
- Deploy services on AWS or Azure

Requirements:
- Python, SQL, LangChain, LangGraph, FastAPI
- Docker, Kubernetes, AWS or Azure
- Vector databases, RAG systems
- Communication, Problem solving, Stakeholder management
"""

MY_PROFILE = {
    "name": "Nikhil",
    "skills": {
        "hard_skills": [
            {"name": "Python",          "proficiency": 8},
            {"name": "LangGraph",        "proficiency": 8},
            {"name": "LangChain",        "proficiency": 8},
            {"name": "FastAPI",          "proficiency": 7},
            {"name": "SQL",              "proficiency": 6},
            {"name": "RAG",              "proficiency": 8},
            {"name": "ChromaDB",         "proficiency": 7},
            {"name": "Pydantic",         "proficiency": 8},
            {"name": "LLM Applications", "proficiency": 7},
        ],
        "soft_skills": [
            {"name": "Problem Solving",  "proficiency": 8},
            {"name": "Communication",    "proficiency": 7},
            {"name": "Collaboration",    "proficiency": 7},
        ]
    },
    "experience_years": 2,
    "projects": [
        {"name": "RoleScout",           "tech_stack": ["Python", "LangGraph", "Pydantic", "LLMs"],          "deployment_status": "internal"},
        {"name": "RAG PDF Search",      "tech_stack": ["LangChain", "ChromaDB", "FastAPI", "Embeddings"],   "deployment_status": "public"},
        {"name": "Agentic Debugger",    "tech_stack": ["LangGraph", "Pydantic", "LLMs", "State Machines"],  "deployment_status": "internal"},
    ],
    "preferred_roles": ["AI Engineer", "Applied AI Engineer", "Forward Deployed Engineer"],
    "location": "Yamunanagar, Haryana"
}

# Run until HITL interrupt
result = graph.invoke(
    {
        "job_desc": JOB_DESCRIPTION,
        "parsed_jd": None,
        "human_feedback": None,
        "final_report": None,
        "profile": MY_PROFILE
    },
    thread
)

parsed = result.get("parsed_jd", {})
print(f"Company:    {parsed.get('company_name')}")
print(f"Role:       {parsed.get('role_title')}")
print(f"Seniority:  {parsed.get('seniority_level')}")
print(f"\nFlags:")
for flag in parsed.get('flags', []):
    print(f"  [{flag['flag_type']}] {flag['desc']}")

### Step 2: Human feedback (optional)

Review the parsed output above.
- If correct → run the next cell as-is (feedback = None)
- If something is wrong → add a correction string, e.g. `"python level should be senior"`

In [ ]:
# Set to None to accept parsed output, or provide a correction string
FEEDBACK = None
# FEEDBACK = "role seniority should be Mid-Level not Senior"

if FEEDBACK:
    graph.update_state(thread, {"human_feedback": FEEDBACK}, as_node="Human Feedback")
else:
    # Clear any previous feedback and continue
    graph.update_state(thread, {"human_feedback": None}, as_node="Human Feedback")

print("Feedback set. Running research agents...")

### Step 3: Resume — run parallel research and synthesize

In [ ]:
# Resume from checkpoint — passes None to continue from where the graph paused
final_result = graph.invoke(None, thread)

print(final_result.get("final_report", "No report generated."))